# Discrete mappings of higher order - exercise lab

This laboratory accompanies [the chapter](https://mvreeuwijk.github.io/chaosbook/disc2d.html). It runs
entirely in your browser (Python via Pyodide) - nothing to install, and any
changes you make are yours alone: re-opening the lab from the chapter resets it.

The full text of the exercises is in the
[chapter's exercise section](https://mvreeuwijk.github.io/chaosbook/disc2d.html#exercises); this notebook
gives you a running start on each one.

In [ ]:
# Run this cell first: it installs the book's package into the
# in-browser Python (takes a few seconds, needs to run once per visit).
%pip install -q chaosbook ipywidgets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact, FloatSlider, IntSlider
import chaosbook as cb

## The standard map at full parameter freedom

The chapter's interactive figure steps through a precomputed grid of $K$
values. Here every slider move iterates the map afresh: pick the kick strength
$K$ and the number of random initial conditions, and watch the islands of
order give way to the sea of chaos.

In [ ]:
def standard_map_explorer(K=2.0, nic=30, nit=400, seed=3):
    rng = np.random.default_rng(seed)
    fig, ax = plt.subplots(figsize=(5, 5))
    for _ in range(nic):
        X = cb.orbit(cb.standard_map, rng.uniform(0, 1, 2), nit, K=K)
        ax.plot(X[:, 0], X[:, 1], ".", markersize=1.5)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")
    ax.set_title(f"standard map, $K={K:.2f}$")
    plt.show()

interact(standard_map_explorer,
         K=FloatSlider(2.0, min=0.0, max=5.0, step=0.05),
         nic=IntSlider(30, min=5, max=100, step=5),
         nit=IntSlider(400, min=50, max=1500, step=50),
         seed=IntSlider(3, min=0, max=20));

## The Hénon map: attractor and Lyapunov exponents

Change $a$ and $b$ and see how the attractor and the two Lyapunov exponents
respond. Use the Jacobian of the map for `cb.lyapunov_spectrum`.

In [ ]:
def henon_jac(p, a, b):
    return np.array([[-2 * a * p[0], 1.0], [b, 0.0]])

def henon_explorer(a=1.4, b=0.3):
    X = cb.orbit(cb.henon, (0.1, 0.3), 5000, a=a, b=b)[100:]
    lam = cb.lyapunov_spectrum(cb.henon, henon_jac, (0.1, 0.3), n=2000, a=a, b=b)
    plt.figure(figsize=(5, 3.5))
    plt.plot(X[:, 0], X[:, 1], "k.", markersize=1)
    plt.xlabel("$x_n$")
    plt.ylabel("$y_n$")
    plt.title(f"$\\Lambda_1={lam[0]:.3f}$, $\\Lambda_2={lam[1]:.3f}$")
    plt.show()

interact(henon_explorer,
         a=FloatSlider(1.4, min=0.5, max=1.5, step=0.01),
         b=FloatSlider(0.3, min=0.0, max=0.5, step=0.01));

## Exercise: Bouncing Ball

Full text: [in the chapter](https://mvreeuwijk.github.io/chaosbook/disc2d.html#exercises). The map is

$$\phi_{n+1} = (\phi_n + r v_n) \bmod 2\pi, \qquad
v_{n+1} = k v_n + (1+k)\cos\phi_{n+1}.$$

Python's `%` operator also works on real numbers. Start with the time series
and the $(\phi_n, v_n)$ plot, then vary $r$ and $k$ (parts a to c).

In [ ]:
def bouncing_ball(s, k, r):
    phi, v = s
    phi = (phi + r * v) % (2 * np.pi)
    return np.array([phi, k * v + (1 + k) * np.cos(phi)])

S = cb.orbit(bouncing_ball, (0.5, 1.0), 300, k=0.3, r=20)
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(11, 3.2))
ax1.plot(S[:, 1], "k.-", linewidth=0.5, markersize=3)
ax1.set_xlabel("$n$"); ax1.set_ylabel("$v_n$")
ax2.plot(S[:, 0], S[:, 1], "k.", markersize=3)
ax2.set_xlabel(r"$\phi_n$"); ax2.set_ylabel("$v_n$")
ax3.plot(S[:-1, 1], S[1:, 1], "k.", markersize=3)
ax3.set_xlabel("$v_n$"); ax3.set_ylabel("$v_{n+1}$")
fig.tight_layout()
plt.show()

Parts d to f: a bifurcation diagram $v_n(r)$ for $k=0.3$, and the stability
of the fixed point with sympy. Zoom in on $r\in[3,3.5]$.

In [ ]:
k = 0.3
rs = np.linspace(1, 10, 600)
pts_r, pts_v = [], []
for r in rs:
    S = cb.orbit(bouncing_ball, (0.5, 1.0), 400, k=k, r=r)
    pts_r.append(np.full(100, r))
    pts_v.append(S[-100:, 1])
plt.figure(figsize=(7, 3.5))
plt.plot(np.concatenate(pts_r), np.concatenate(pts_v), "k.", markersize=1)
plt.xlabel("$r$"); plt.ylabel("$v_n$")
plt.show()

# stability of a fixed point: symbolic Jacobian of the map
import sympy as sp
phi, v, ks, rs_ = sp.symbols("phi v k r")
f = sp.Matrix([phi + rs_ * v, ks * v + (1 + ks) * sp.cos(phi + rs_ * v)])
J = f.jacobian([phi, v])
J

## Exercise: Dripping Faucet Model

Full text: [in the chapter](https://mvreeuwijk.github.io/chaosbook/disc2d.html#exercises). Parameters: $m^c=1$, $\delta=7$,
$\omega=60\pi$, $\mu=3$. Below, $y$ and $v$ are the damped-oscillator
solution; complete the map for $T_n$, $m^0_n$, $y^0_n$, $v^0_n$ (parts a, b).

In [ ]:
mc, delta, omega, mu = 1.0, 7.0, 60 * np.pi, 3.0

def y(t, y0, v0):
    return np.exp(-delta * t) / omega * (
        (omega * np.cos(omega * t) + delta * np.sin(omega * t)) * y0
        + np.sin(omega * t) * v0)

def v(t, y0, v0):
    return np.exp(-delta * t) / omega * (
        -(omega**2 + delta**2) * np.sin(omega * t) * y0
        + (omega * np.cos(omega * t) - delta * np.sin(omega * t)) * v0)

def faucet(s, mu=mu):
    T, m0, y0, v0 = s
    T = (mc - m0) / mu
    return np.array([T, mc / 2 - y(T, y0, v0) / 9, 1.0, v(T, y0, v0)])

S = cb.orbit(faucet, (0.0, mc / 2, 1.0, 0.0), 200)
T = S[1:, 0]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3.2))
ax1.plot(T, "k.-", linewidth=0.5, markersize=3)
ax1.set_xlabel("$n$"); ax1.set_ylabel("$T_n$")
ax2.plot(T[:-1], T[1:], "k.", markersize=3)
ax2.set_xlabel("$T_n$"); ax2.set_ylabel("$T_{n+1}$")
fig.tight_layout()
plt.show()

Parts c and d: an effective Lyapunov exponent from two series that differ by
$\epsilon=10^{-9}$ in $v^0_1$. Use higher-precision arithmetic if you need
more digits (for example `mpmath` with `mp.mp.dps = 20`). For part d, compare
the drip times $T_n$ (column 0 of the state) instead of $v^0_n$ (column 3).

In [ ]:
eps = 1e-9
S1 = cb.orbit(faucet, (0.0, mc / 2, 1.0, 0.0), 40)
S2 = cb.orbit(faucet, (0.0, mc / 2, 1.0, eps), 40)
d = np.abs(S2[:, 3] - S1[:, 3])
plt.semilogy(d, "k.-")
plt.xlabel("$n$"); plt.ylabel(r"$|\tilde v - v|$")
plt.show()

## Exercise: Kicked U-tube

Full text: [in the chapter](https://mvreeuwijk.github.io/chaosbook/disc2d.html#exercises). Between kicks the water level
follows a damped oscillation with coefficients $A$, $B$, $C$; at each kick the
level jumps by $-K\,\mathrm{sgn}(\dot\delta)$. Use `np.where(v >= 0, 1, -1)`
so that $\mathrm{sgn}(0)=1$.

In [ ]:
def tube(s, mu=0.01, K=0.1, phi=0.76):
    d, dd = s
    tau = 2 * np.pi * phi
    w = np.sqrt(1 - mu**2)
    e = np.exp(-mu * tau)
    A = e * (mu * np.sin(w * tau) + w * np.cos(w * tau)) / w
    B = e * np.sin(w * tau) / w
    C = -e * (mu * np.sin(w * tau) - w * np.cos(w * tau)) / w
    dd_new = -B * d + C * dd
    sgn = np.where(dd_new >= 0, 1.0, -1.0)
    return np.array([A * d + B * dd - K * sgn, dd_new])

# parts c and f: phase plane after the transients are gone
fig, ax = plt.subplots(figsize=(4.2, 4.2))
for phi in (0.3, 0.55, 0.76):
    S = cb.orbit(tube, (0.2, 0.0), 3000, phi=phi)[1000:]
    ax.plot(S[:, 0], S[:, 1], ".", markersize=2, label=f"$\\phi={phi}$")
ax.set_xlabel(r"$\delta_n$"); ax.set_ylabel(r"$\dot\delta_n$")
ax.legend(frameon=False)
plt.show()

In [ ]:
# part d: bifurcation diagram over phi in [0, 1] at K = 0.1, mu = 0.01
phis = np.linspace(0.005, 1, 300)
pp, dv = [], []
for phi in phis:
    S = cb.orbit(tube, (0.2, 0.0), 1300, phi=phi)[-100:]
    pp.append(np.full(100, phi)); dv.append(S[:, 0])
plt.figure(figsize=(7, 3.5))
plt.plot(np.concatenate(pp), np.concatenate(dv), "k.", markersize=1)
plt.xlabel(r"$\phi$"); plt.ylabel(r"$\delta_n$")
plt.show()